In [ ]:
# Spatial Domain DenseNet121 - Standard Architecture
# Metrics: Accuracy, Cohen's Kappa, Precision, Recall, F1, Specificity, Error Rate

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report
)
import os
import warnings
import gc
warnings.filterwarnings('ignore')

# ================== Device Setup ==================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ================== Step 1: Custom Dataset for Fruits-360 ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360 - Spatial Domain"""

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((
                            os.path.join(class_dir, img_name),
                            self.class_to_idx[class_name]
                        ))

        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label


def load_fruits_dataset(data_root):
    """Load Fruits-360 dataset with standard spatial transforms"""

    # Standard ImageNet normalization for DenseNet121
    transform_train = transforms.Compose([
        transforms.Resize((100, 100)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    transform_test = transforms.Compose([
        transforms.Resize((100, 100)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dir = os.path.join(data_root, 'Training')
    test_dir  = os.path.join(data_root, 'Test')

    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset  = FruitsDataset(test_dir,  transform=transform_test)

    return trainset, testset, trainset.classes

# ================== Step 2: Standard DenseNet121 Model ==================

class SpatialDomainDenseNet121(nn.Module):
    """
    Standard DenseNet121 for spatial domain classification.
    Uses the original 3-channel RGB input with a custom classifier head.
    """

    def __init__(self, num_classes, dropout_rate=0.5):
        super(SpatialDomainDenseNet121, self).__init__()

        # Load pretrained DenseNet121 (standard architecture, no modifications)
        self.densenet = models.densenet121(pretrained=True)

        # Number of features from the DenseNet121 feature extractor
        num_features = self.densenet.classifier.in_features  # 1024

        # Replace the original single-layer classifier with a deeper head
        self.densenet.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )

        self._initialize_classifier_weights()

    def _initialize_classifier_weights(self):
        """Xavier initialization for the new classifier layers"""
        for m in self.densenet.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.densenet(x)

# ================== Step 3: Early Stopping ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""

    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience        = patience
        self.min_delta       = min_delta
        self.verbose         = verbose
        self.counter         = 0
        self.best_score      = None
        self.early_stop      = False
        self.best_model_state = None

    def __call__(self, val_accuracy, model):
        score = val_accuracy

        if self.best_score is None:
            self.best_score       = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_score       = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter          = 0

# ================== Step 4: Training Function ==================

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, weight_decay=1e-4):
    """
    Train SpatialDomainDenseNet121 with differential learning rates:
      - Pretrained backbone  : lr * 0.01
      - New classifier head  : lr * 0.5
    """

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Separate pretrained backbone params from new classifier params
    pretrained_params = []
    new_params        = []

    for name, param in model.named_parameters():
        if 'densenet.classifier' in name:
            new_params.append(param)
        else:
            pretrained_params.append(param)

    optimizer = torch.optim.AdamW([
        {'params': pretrained_params, 'lr': lr * 0.01},
        {'params': new_params,        'lr': lr * 0.5}
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )

    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)

    train_losses      = []
    val_losses        = []
    train_accuracies  = []
    val_accuracies    = []

    best_val_accuracy = 0.0
    best_model_state  = None

    # Mixed precision scaler (only used on CUDA)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    for epoch in range(epochs):

        # ---- Training phase ----
        model.train()
        running_loss         = 0.0
        correct_train        = 0
        total_train          = 0
        num_batches_processed = 0

        train_pbar = tqdm(train_loader,
                          desc=f"Epoch {epoch+1}/{epochs} [Train]",
                          leave=False)

        for i, (images, labels) in enumerate(train_pbar):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Skip corrupted batches
            if torch.isnan(images).any() or torch.isinf(images).any():
                print(f"Warning: NaN/Inf in input batch {i}, skipping...")
                continue

            optimizer.zero_grad(set_to_none=True)

            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss    = criterion(outputs, labels)

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss in batch {i}, skipping...")
                    continue

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss in batch {i}, skipping...")
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            running_loss          += loss.item()
            num_batches_processed += 1
            _, predicted           = torch.max(outputs.data, 1)
            total_train           += labels.size(0)
            correct_train         += (predicted == labels).sum().item()

            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc':  f'{100 * correct_train / total_train:.2f}%'
            })

            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()

        scheduler.step()

        if num_batches_processed == 0:
            print(f"Epoch {epoch+1}: No batches processed, skipping...")
            continue

        avg_train_loss   = running_loss / num_batches_processed
        train_accuracy   = 100 * correct_train / total_train if total_train > 0 else 0.0
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)

        # ---- Validation phase ----
        model.eval()
        running_val_loss = 0.0
        correct          = 0
        total            = 0

        with torch.no_grad():
            val_pbar = tqdm(val_loader,
                            desc=f"Epoch {epoch+1}/{epochs} [Val]",
                            leave=False)

            for images, labels in val_pbar:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                if scaler is not None:
                    with torch.amp.autocast('cuda'):
                        outputs = model(images)
                        loss    = criterion(outputs, labels)
                else:
                    outputs = model(images)
                    loss    = criterion(outputs, labels)

                running_val_loss += loss.item()
                _, predicted      = torch.max(outputs.data, 1)
                total            += labels.size(0)
                correct          += (predicted == labels).sum().item()

                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc':  f'{100 * correct / total:.2f}%'
                })

        avg_val_loss  = running_val_loss / len(val_loader)
        val_accuracy  = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val   Loss: {avg_val_loss:.4f}, Val   Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}')

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")

    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 5: Comprehensive Metrics Computation ==================

def compute_all_metrics(all_labels, all_predictions, num_classes):
    """
    Compute all requested performance metrics:
      - Overall Accuracy
      - Cohen's Kappa Score
      - Macro-averaged Precision
      - Macro-averaged Recall (Sensitivity)
      - Macro-averaged F1 Score
      - Overall Specificity  (macro-averaged across classes)
      - Overall Error Rate

    Parameters
    ----------
    all_labels      : list/array of ground-truth integer class indices
    all_predictions : list/array of predicted integer class indices
    num_classes     : total number of classes

    Returns
    -------
    metrics : dict with all metric names and values
    """

    all_labels      = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # --- Core sklearn metrics ---
    accuracy  = accuracy_score(all_labels, all_predictions)
    kappa     = cohen_kappa_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions,
                                average='macro', zero_division=0)
    recall    = recall_score(all_labels, all_predictions,
                             average='macro', zero_division=0)
    f1        = f1_score(all_labels, all_predictions,
                         average='macro', zero_division=0)

    # --- Specificity (macro-averaged) ---
    # For each class c:
    #   TN_c = samples not in class c that were correctly NOT predicted as c
    #   FP_c = samples not in class c that were wrongly predicted as c
    #   specificity_c = TN_c / (TN_c + FP_c)
    cm = confusion_matrix(all_labels, all_predictions, labels=list(range(num_classes)))

    specificities = []
    for c in range(num_classes):
        tp_c = cm[c, c]
        fn_c = cm[c, :].sum() - tp_c          # row c, excluding diagonal
        fp_c = cm[:, c].sum() - tp_c          # col c, excluding diagonal
        tn_c = cm.sum() - tp_c - fn_c - fp_c  # everything else

        denom = tn_c + fp_c
        spec_c = tn_c / denom if denom > 0 else 0.0
        specificities.append(spec_c)

    specificity = np.mean(specificities)

    # --- Error rate ---
    error_rate = 1.0 - accuracy

    metrics = {
        'Overall Accuracy (%)':       accuracy  * 100,
        "Cohen's Kappa Score":         kappa,
        'Overall Precision (macro)':  precision,
        'Overall Recall (macro)':     recall,
        'Overall F1 Score (macro)':   f1,
        'Overall Specificity (macro)': specificity,
        'Overall Error Rate (%)':     error_rate * 100,
    }

    return metrics, cm


def print_metrics(metrics, cm, classes):
    """Pretty-print all computed metrics and the confusion matrix summary"""

    print("\n" + "=" * 60)
    print("         OVERALL PERFORMANCE METRICS")
    print("=" * 60)

    for metric_name, value in metrics.items():
        if '%' in metric_name:
            print(f"  {metric_name:<35s}: {value:.4f}%")
        else:
            print(f"  {metric_name:<35s}: {value:.6f}")

    print("=" * 60)

    # Per-class accuracy from the diagonal of the confusion matrix
    print("\nPer-Class Accuracy (top-10 and bottom-10 by accuracy):")
    per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    sorted_idx    = np.argsort(per_class_acc)[::-1]

    print(f"\n  {'Class':<40s} {'Acc (%)':>8}")
    print(f"  {'-'*40} {'-'*8}")

    top_n = min(10, len(classes))
    print("  [Top classes]")
    for i in sorted_idx[:top_n]:
        print(f"  {classes[i]:<40s} {per_class_acc[i]*100:>7.2f}%")

    print("  [Bottom classes]")
    for i in sorted_idx[-top_n:]:
        print(f"  {classes[i]:<40s} {per_class_acc[i]*100:>7.2f}%")

    print("=" * 60)

# ================== Step 6: Visualization ==================

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation loss / accuracy curves"""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Spatial Domain DenseNet121 - Training Curves',
                 fontsize=14, fontweight='bold')

    epochs = range(1, len(train_losses) + 1)

    ax1.plot(epochs, train_losses, 'b-', label='Training Loss',   linewidth=2)
    ax1.plot(epochs, val_losses,   'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss',  fontsize=12)
    ax1.set_title('Loss Curves', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy',   linewidth=2)
    ax2.plot(epochs, val_accuracies,   'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch',        fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Accuracy Curves', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix_summary(cm, classes, top_n=20):
    """
    Plot a condensed confusion matrix for the top_n most-confused classes.
    Plotting the full matrix for 100+ classes is unreadable.
    """

    # Select top_n classes with the most off-diagonal errors
    errors_per_class = cm.sum(axis=1) - cm.diagonal()
    top_indices      = np.argsort(errors_per_class)[::-1][:top_n]

    cm_subset     = cm[np.ix_(top_indices, top_indices)]
    class_names   = [classes[i] for i in top_indices]

    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(cm_subset, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(top_n))
    ax.set_yticks(range(top_n))
    ax.set_xticklabels(class_names, rotation=90, fontsize=7)
    ax.set_yticklabels(class_names, fontsize=7)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label',      fontsize=11)
    ax.set_title(f'Confusion Matrix (Top {top_n} Most-Confused Classes)',
                 fontsize=13, fontweight='bold')

    thresh = cm_subset.max() / 2.0
    for i in range(top_n):
        for j in range(top_n):
            ax.text(j, i, str(cm_subset[i, j]),
                    ha='center', va='center', fontsize=6,
                    color='white' if cm_subset[i, j] > thresh else 'black')

    plt.tight_layout()
    plt.show()


def plot_metrics_bar(metrics):
    """Bar chart of all scalar metrics (excluding % ones for a clean 0-1 scale)"""

    # Separate percentage metrics from 0-1 metrics
    pct_metrics = {k: v for k, v in metrics.items() if '%' in k}
    raw_metrics = {k: v for k, v in metrics.items() if '%' not in k}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Spatial Domain DenseNet121 - Performance Summary',
                 fontsize=14, fontweight='bold')

    # 0–1 scale
    ax = axes[0]
    names  = list(raw_metrics.keys())
    values = list(raw_metrics.values())
    colors = ['steelblue', 'seagreen', 'darkorange', 'mediumpurple', 'crimson']
    bars   = ax.bar(names, values, color=colors[:len(names)], edgecolor='black', alpha=0.85)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score (0 – 1)', fontsize=11)
    ax.set_title('Classification Metrics (0–1 scale)', fontsize=12, fontweight='bold')
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Percentage scale
    ax2 = axes[1]
    pct_names  = list(pct_metrics.keys())
    pct_values = list(pct_metrics.values())
    pct_colors = ['steelblue', 'crimson']
    bars2 = ax2.bar(pct_names, pct_values, color=pct_colors[:len(pct_names)],
                    edgecolor='black', alpha=0.85)
    ax2.set_ylim(0, 110)
    ax2.set_ylabel('Percentage (%)', fontsize=11)
    ax2.set_title('Percentage Metrics', fontsize=12, fontweight='bold')
    ax2.set_xticklabels(pct_names, rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars2, pct_values):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

# ================== Step 7: Main Pipeline ==================

def main():
    print("=" * 70)
    print("  Spatial Domain DenseNet121 - Standard Architecture")
    print("  Metrics: Accuracy | Kappa | Precision | Recall |")
    print("           F1 | Specificity | Error Rate")
    print("=" * 70)

    # ── Dataset path ──────────────────────────────────────────────────────────
    data_root = r'C:\Users\CSE_SDPL\Downloads\fruits-360_100x100\fruits-360'

    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        print("Please update 'data_root' to point to your dataset location.")
        return

    # ── Step 1: Load dataset ──────────────────────────────────────────────────
    print("\n[Step 1] Loading Fruits-360 dataset (spatial domain)...")
    try:
        trainset, testset, classes = load_fruits_dataset(data_root)
    except Exception as e:
        print(f"ERROR loading dataset: {e}")
        return

    num_classes = len(classes)
    print(f"Number of classes: {num_classes}")

    # ── Step 2: Train / validation split ─────────────────────────────────────
    print("\n[Step 2] Splitting training set into train / validation (85/15)...")
    train_size   = int(0.85 * len(trainset))
    val_size     = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"Training   samples: {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")
    print(f"Test       samples: {len(testset)}")

    # ── Step 3: DataLoaders ───────────────────────────────────────────────────
    # num_workers = 0 on Windows to avoid multiprocessing issues
    num_workers = 4 if os.name != 'nt' else 0
    batch_size  = 64        # Conservative; increase if GPU memory allows

    print(f"\n[Step 3] Creating DataLoaders (batch={batch_size}, workers={num_workers})...")

    train_loader = DataLoader(
        train_subset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    val_loader = DataLoader(
        val_subset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    test_loader = DataLoader(
        testset, batch_size=batch_size, shuffle=False,
        num_workers=0       # single-process for reproducible ordering
    )

    # ── Step 4: Model ─────────────────────────────────────────────────────────
    print("\n[Step 4] Initialising Spatial Domain DenseNet121...")
    model = SpatialDomainDenseNet121(
        num_classes=num_classes, dropout_rate=0.5
    ).to(device)

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters     : {total_params:,}")
    print(f"Trainable parameters : {trainable_params:,}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory allocated : {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

    # ── Step 5: Training ──────────────────────────────────────────────────────
    print("\n[Step 5] Training model...")
    try:
        train_losses, val_losses, train_accuracies, val_accuracies = train_model(
            model, train_loader, val_loader,
            epochs=50, lr=0.001, weight_decay=5e-4
        )
    except Exception as e:
        print(f"\nERROR during training: {e}")
        import traceback
        traceback.print_exc()
        return

    # ── Step 5.1: Training curves ─────────────────────────────────────────────
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)

    # ── Step 6: Evaluation on test set ────────────────────────────────────────
    print("\n[Step 6] Evaluating on test set and computing all metrics...")
    model.eval()

    all_predictions = []
    all_labels      = []
    all_confidences = []

    correct = 0
    total   = 0

    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for images, labels in test_pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs      = model(images)
            probabilities = F.softmax(outputs, dim=1)
            confidences, predicted = torch.max(probabilities, 1)

            total   += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_confidences.extend(confidences.cpu().numpy())

            test_pbar.set_postfix({'acc': f'{100 * correct / total:.2f}%'})

    # ── Step 7: Compute & display metrics ─────────────────────────────────────
    print("\n[Step 7] Computing comprehensive performance metrics...")
    metrics, cm = compute_all_metrics(all_labels, all_predictions, num_classes)
    print_metrics(metrics, cm, classes)

    # ── Step 8: Visualisations ────────────────────────────────────────────────
    print("\n[Step 8] Generating metric visualisations...")
    plot_metrics_bar(metrics)
    plot_confusion_matrix_summary(cm, classes, top_n=min(20, num_classes))

    # ── Step 9: Save model ────────────────────────────────────────────────────
    print("\n[Step 9] Saving trained model...")
    save_path = 'fruits_spatial_densenet121.pth'
    try:
        torch.save({
            'model_state_dict': model.state_dict(),
            'metrics':          metrics,
            'classes':          classes,
            'num_classes':      num_classes,
            'architecture':     'SpatialDomainDenseNet121'
        }, save_path)
        print(f"Model saved as '{save_path}'")
    except Exception as e:
        print(f"ERROR saving model: {e}")

    # ── Final summary ─────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("  FINAL SUMMARY")
    print("=" * 70)
    for k, v in metrics.items():
        if '%' in k:
            print(f"  {k:<35s}: {v:.4f}%")
        else:
            print(f"  {k:<35s}: {v:.6f}")
    print("=" * 70)
    print("Pipeline completed successfully!")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if __name__ == "__main__":
    main()